# Entity
>  A lightweight, _immutable_ wrapper around a JSON-LD dict.

 **Key feature:**

* Stores *content* (your domain data) plus a **composer-built context**.
* Exposes `.normalized()` which returns a deterministic N-Quads string using URDNA2015—ideal for hashing or signing later.
* Internally caches the normalisation so repeated calls are free.

In [ ]:
#| default_exp core.entity

In [ ]:
#| export
from __future__ import annotations

from fastcore.basics import patch
import hashlib, json
from datetime import datetime
from functools import cached_property
from typing import Any, Dict, List, Optional
from copy import deepcopy  # Add this import!
import uuid

from pydantic import BaseModel, Field, model_validator

from cogitarelink.core.debug import get_logger
from cogitarelink.core.context import ContextProcessor
from cogitarelink.vocab.composer import composer

log = get_logger("entity")
_proc = ContextProcessor()

## hash helper  

In [ ]:
#| export
def _deep_hash(d: Dict[str, Any]) -> str:
    "SHA-256 over canonical JSON (sort keys, no spaces)."
    return hashlib.sha256(
        json.dumps(d, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()

## Entity model  

In [ ]:
#| export
# helper reused by the property
def _extract_children(data):
    kids=[]
    for k,v in list(data.items()):
        if isinstance(v, dict) and ("@type" in v or "type" in v):
            kids.append(v); data.pop(k)
        elif isinstance(v, list):
            for item in list(v):
                if isinstance(item, dict) and ("@type" in item or "type" in item):
                    kids.append(item); v.remove(item)
    return kids

In [ ]:
#| export
class Entity(BaseModel):
    """
    Immutable view of a JSON-LD resource with fixes for the identified issues.
    
    Parameters
    ----------
    id      : Optional IRI (string).  If omitted a blank node is used.
    vocab   : list of registry prefixes that define the entity's context.
              The order matters: first prefix is considered primary.
    content : dict with user-supplied properties (e.g. {"name":"Ada"}).
    meta    : optional metadata (timestamps, provenance).
    """

    id:      Optional[str]          = Field(default=None, alias="@id")
    vocab:   List[str]              = Field(min_items=1)
    content: Dict[str, Any]         = Field(default_factory=dict)
    meta:    Dict[str, Any] | None  = None
    created: datetime               = Field(default_factory=datetime.utcnow)

    model_config = dict(frozen=True, populate_by_name=True)
    
    # ------------------------------------------------------------------
    # validators
    # ------------------------------------------------------------------
    @model_validator(mode="after")
    def _attach_context(cls, v):
        # Compose once and store in private attr _ctx
        ctx = composer.compose(v.vocab)["@context"]
        object.__setattr__(v, "_ctx", ctx)
        # Make a deep copy of content to ensure immutability
        immutable_content = deepcopy(v.content)
        object.__setattr__(v, "content", immutable_content)
        return v

    # ------------------------------------------------------------------
    # public helpers
    # ------------------------------------------------------------------
    @property
    def as_json(self) -> Dict[str, Any]:
        "Full JSON-LD dict with `@context`"
        base = {"@context": self._ctx}
        if self.id:
            base["@id"] = self.id
        base.update(self.content)
        return base  # Return the constructed JSON-LD object

    # ------------------------------------------------------------------
    # expensive calculation cached per-instance
    # ------------------------------------------------------------------
    @cached_property
    def normalized(self) -> str:
        "URDNA2015 N-Quads representation."
        return _proc.normalize(self.as_json)

    # convenience ------------------------------------------------------
    @property
    def sha256(self) -> str:
        "Digest over `normalized` property."
        return hashlib.sha256(self.normalized.encode()).hexdigest()

    # Override __getattribute__ to return copies of mutable objects
    def __getattribute__(self, name):
        attr = super().__getattribute__(name)
        if name == 'content' and isinstance(attr, dict):
            return deepcopy(attr)
        return attr
    

    # Apply patch first
    @cached_property # Then apply cached_property
    def children(self: Entity) -> List["Entity"]:
        "Child Entities whose dicts contain an explicit @type."
        # Ensure we work on a copy to avoid modifying the original content
        content_copy = deepcopy(self.content)
        kids_raw = _extract_children(content_copy)
        # Pass the parent's vocab to the children
        return [Entity(vocab=self.vocab, content=k) for k in kids_raw]

## Tests

In [ ]:
#| hide
from fastcore.test import test_eq, ExceptionExpected

# Create an entity instance
e = Entity(vocab=["schema"], content={"name": "Ada"})

# Test the normalized property (access as property, not method)
test_eq(isinstance(e.normalized, str), True)
test_eq(e.normalized, e.normalized)  # Should be identical due to caching

# Test the sha256 property
h1 = e.sha256
h2 = e.sha256
test_eq(h1, h2)  # Should be identical hashes

# Verify JSON-LD structure is correct
json_ld = e.as_json
test_eq("@context" in json_ld, True)
test_eq(json_ld["name"], "Ada")

# Test immutability of internal content
original_content = e.content
try:
    # This should not modify the original entity's content
    content_copy = e.content
    content_copy["age"] = 10
    # Verify the entity's content is unchanged
    test_eq("age" in e.content, False)
except Exception as ex:
    print(f"Error: {ex}")


In [ ]:
#| export
@model_validator(mode="after")
def _attach_context(cls, v):
    # Generate blank-node ID if missing
    if v.id is None:
        object.__setattr__(v, "id", f"urn:uuid:{uuid.uuid4()}")
    ctx = composer.compose(v.vocab)["@context"]
    object.__setattr__(v, "_ctx", ctx)
    return v

In [ ]:
parent = Entity(vocab=["schema"], content={"name":"A","child":{"@type":"Person","name":"B"}})
assert len(parent.children)==1

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()